# Model precision characterisation

Two experiments characterise the limits of what the waveform optimiser can achieve:

**Experiment A — Amplitude sweep**
Brief (10 ms) rectangular pulse at a fixed position; amplitude swept across the deliverable
range. For each level: N_RUNS simulations at 1 ms bins → FWHM of onset response
(rise HWHM + fall HWHM), decay HWHM after pulse offset, and std of per-run peak rates.
Reveals how temporal precision and variability scale with evoked firing rate.

**Experiment B — Background current sweep**
Same brief pulse at a fixed moderate amplitude; background drive `I_bg_exc_pA` swept
to set different pre-existing baseline rates. Same metrics as Exp A, but now as a
function of how active the network already is before the stimulus arrives.

*Note: with 8000 exc + 2000 inh neurons each simulation takes ~30–60 s.
Reduce `N_RUNS` or `N_AMP_LEVELS` / `N_BG_LEVELS` for quick exploratory runs.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use(Path('..') / 'configs' / 'mpl.mplstyle')

from designer_waveform.models import RandomEINetwork, load_config
from designer_waveform.optics import OpticsConfig, SigmoidPowerCurve
from designer_waveform.waveforms import RectangularPulseWaveform

In [ ]:
# ── Probe pulse ───────────────────────────────────────────────────────────
PULSE_ONSET_MS = 80.0    # onset within stim window — well clear of edges
PULSE_DUR_MS   = 10.0    # brief pulse so onset and decay don't overlap
STIM_DUR_MS    = 280.0   # total window: gives 80 ms pre-pulse + 190 ms post-pulse tail

# ── Binning ───────────────────────────────────────────────────────────────
FINE_BIN_MS = 1.0        # 1 ms bins — well below the ~10–30 ms features we're measuring

# ── Network ──────────────────────────────────────────────────────────────
N_EXC     = 8000
N_INH     = 2000
T_PRE_MS  = 200.0
T_POST_MS = 100.0

# ── Multi-run settings ───────────────────────────────────────────────────
N_RUNS    = 10           # per condition — increase to 20+ for publication-quality error bars
SEED_BASE = 1000
VARY_INIT_V       = True
VARY_CONNECTIVITY = True
VARY_WEIGHTS      = True

# ── Experiment A: amplitude sweep ─────────────────────────────────────────
# Pulse amplitude is varied to evoke a range of firing rates.
AMP_LEVELS_MW = None     # set to a list to override auto-generation
N_AMP_LEVELS  = 8
MIN_AMP_MW    = 0.3      # lower bound (mW)

# ── Experiment B: background current sweep ───────────────────────────────
# I_bg_exc_pA is varied to set different pre-existing baseline rates.
# Probe pulse is fixed at BG_PULSE_AMP_MW.
BG_LEVELS_PA    = None   # set to list to override, e.g. [50, 100, 150, 200, 250, 300]
N_BG_LEVELS     = 6
MIN_BG_PA       = 50.0
MAX_BG_PA       = 300.0
BG_PULSE_AMP_MW = 2.0    # probe amplitude for the background sweep (mW)

# ── Opsin ─────────────────────────────────────────────────────────────────
# 'c1v1'    : C1V1-A  (i_max ≈ 1175 pA, K½ ≈ 0.0015 mW/mm²)
# 'chrmine' : ChRmine (i_max ≈ 4600 pA, K½ ≈ 0.037  mW/mm²)
OPSIN = 'c1v1'

## Build model

In [ ]:
CONFIG_PATH = Path('..') / 'configs' / 'random_ei.json'
cfg = load_config(CONFIG_PATH)
cfg.N_exc       = N_EXC
cfg.N_inh       = N_INH
cfg.t_pre_ms    = T_PRE_MS
cfg.t_post_ms   = T_POST_MS
cfg.t_stim_ms   = STIM_DUR_MS
cfg.psth_bin_ms = FINE_BIN_MS

_optics = OpticsConfig.from_file(Path('..') / 'data' / 'optics_params.json')
_curve  = {'c1v1': SigmoidPowerCurve.c1v1, 'chrmine': SigmoidPowerCurve.chrmine}[OPSIN]()
model   = RandomEINetwork(cfg, optics=_optics, power_curve=_curve,
                          normalization='max_expression')

MAX_POWER_MW = _optics.area_mm2 * 0.1 / _optics.total_transmission

OUTPUT_DIR = Path('../results') / 'precision'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Model built.  N_exc={N_EXC}, N_inh={N_INH}')
print(f'Opsin: {OPSIN}  i_max={_curve.i_max_pA:.0f} pA  K½={_curve.half_sat_mW_mm2} mW/mm²')
print(f'Max deliverable power : {MAX_POWER_MW:.2f} mW')
print(f'Opsin mean            : {model._stim_dist_pA.mean():.1f} pA  '
      f'frac zero: {(model._stim_dist_pA == 0).mean():.3f}')
print(f'Stim window           : {STIM_DUR_MS:.0f} ms  ({FINE_BIN_MS:.1f} ms bins)')
print(f'Output dir            : {OUTPUT_DIR}')

# Resolve sweep levels
amp_levels = (np.array(AMP_LEVELS_MW, dtype=float) if AMP_LEVELS_MW is not None
              else np.linspace(MIN_AMP_MW, MAX_POWER_MW, N_AMP_LEVELS))
bg_levels  = (np.array(BG_LEVELS_PA, dtype=float) if BG_LEVELS_PA is not None
              else np.linspace(MIN_BG_PA, MAX_BG_PA, N_BG_LEVELS))

print(f'\nAmplitude levels ({len(amp_levels)}): '
      + ', '.join(f'{a:.2f}' for a in amp_levels) + ' mW')
print(f'Background levels ({len(bg_levels)}): '
      + ', '.join(f'{b:.0f}' for b in bg_levels) + ' pA')

In [ ]:
def measure_pulse_response(t_ms, mean_hz, pulse_onset_ms, pulse_dur_ms,
                            pre_buffer_ms=15.0):
    """
    Extract temporal precision metrics from a single-pulse mean PSTH.

    Metrics returned (all times in ms, rates in Hz):
      baseline_hz    : mean rate in the pre-pulse window
      peak_hz        : absolute peak rate (baseline + peak_above)
      peak_above_hz  : peak rate above baseline
      peak_t_ms      : time of peak
      latency_ms     : time from pulse onset to peak
      rise_hwhm_ms   : time from half-max (rising) to peak
      fall_hwhm_ms   : time from peak to half-max (falling)
      fwhm_ms        : total FWHM = rise_hwhm + fall_hwhm
      decay_hwhm_ms  : time from pulse offset to when rate drops to half its
                       offset-time value above baseline (the 'switch-off' speed)

    Returns None if no measurable response above baseline.
    """
    pre_mask = t_ms < (pulse_onset_ms - pre_buffer_ms)
    baseline = float(mean_hz[pre_mask].mean()) if pre_mask.any() else 0.0
    above    = mean_hz - baseline

    post_mask = t_ms >= pulse_onset_ms
    if not post_mask.any():
        return None
    t_post = t_ms[post_mask]
    a_post = above[post_mask]

    peak_i = int(np.argmax(a_post))
    peak_a = float(a_post[peak_i])
    peak_t = float(t_post[peak_i])
    if peak_a <= 0:
        return None
    half = peak_a / 2.0

    def _cross_up(t, a, thresh):
        idxs = np.where(np.diff((a >= thresh).astype(int)) > 0)[0]
        if not len(idxs):
            return float(t[0])
        i = idxs[-1]
        return float(np.interp(thresh, [a[i], a[i + 1]], [t[i], t[i + 1]]))

    def _cross_dn(t, a, thresh):
        idxs = np.where(np.diff((a >= thresh).astype(int)) < 0)[0]
        if not len(idxs):
            return float(t[-1])
        i = idxs[0]
        return float(np.interp(thresh, [a[i + 1], a[i]], [t[i + 1], t[i]]))

    t_rise = _cross_up(t_post[:peak_i + 1], a_post[:peak_i + 1], half)
    t_fall = _cross_dn(t_post[peak_i:],     a_post[peak_i:],     half)

    # Decay after offset: half-time for activity to drop from its offset-time
    # level back to baseline (captures the network's 'switch-off' speed)
    offset_ms = pulse_onset_ms + pulse_dur_ms
    offset_i  = int(np.searchsorted(t_post, offset_ms))
    decay_hwhm = np.nan
    if offset_i < len(a_post) - 1:
        a_dec = a_post[offset_i:]
        t_dec = t_post[offset_i:]
        level = float(a_dec[0])
        if level > 0:
            decay_hwhm = _cross_dn(t_dec, a_dec, level / 2.0) - offset_ms

    return {
        'baseline_hz':   baseline,
        'peak_hz':       baseline + peak_a,
        'peak_above_hz': peak_a,
        'peak_t_ms':     peak_t,
        'latency_ms':    peak_t - pulse_onset_ms,
        'rise_hwhm_ms':  peak_t - t_rise,
        'fall_hwhm_ms':  t_fall - peak_t,
        'fwhm_ms':       t_fall - t_rise,
        'decay_hwhm_ms': decay_hwhm,
    }

print('measure_pulse_response() defined.')

## Experiment A — Amplitude sweep

Varies pulse amplitude to evoke a range of peak firing rates.
Measures FWHM components and rate variability as a function of evoked rate.

In [ ]:
print(f'Sweep A: {len(amp_levels)} amplitude levels × {N_RUNS} runs  '
      f'({PULSE_DUR_MS:.0f} ms pulse at t={PULSE_ONSET_MS:.0f} ms, {FINE_BIN_MS:.1f} ms bins)')

amp_results = []
t_fine = None   # set from first run

for i_a, amp in enumerate(amp_levels):
    wf = RectangularPulseWaveform(onset_ms=PULSE_ONSET_MS, duration_ms=PULSE_DUR_MS,
                                   amplitude=amp)
    _runs = []
    for _i in range(N_RUNS):
        _r = model.run(wf, seed=SEED_BASE + _i,
                       vary_init_v=VARY_INIT_V,
                       vary_connectivity=VARY_CONNECTIVITY,
                       vary_weights=VARY_WEIGHTS)
        _runs.append(_r['psth_exc'] / (FINE_BIN_MS / 1000.0))
    if t_fine is None:
        t_fine = _r['t_psth_ms']

    _arr     = np.stack(_runs)
    mean_hz  = _arr.mean(0)
    sem_hz   = _arr.std(0) / np.sqrt(N_RUNS)

    # Per-run peak rate in the post-pulse window (captures trial variability)
    post_mask      = t_fine >= PULSE_ONSET_MS
    per_run_peaks  = _arr[:, post_mask].max(1) if post_mask.any() else np.zeros(N_RUNS)

    m = measure_pulse_response(t_fine, mean_hz, PULSE_ONSET_MS, PULSE_DUR_MS)

    amp_results.append({
        'amp_mw':        amp,
        'mean_hz':       mean_hz,
        'sem_hz':        sem_hz,
        'per_run_peaks': per_run_peaks,
        'peak_mean_hz':  float(per_run_peaks.mean()),
        'peak_std_hz':   float(per_run_peaks.std()),
        'metrics':       m,
    })
    tag = (f'FWHM={m["fwhm_ms"]:.1f} ms  decay={m["decay_hwhm_ms"]:.1f} ms'
           if m and not np.isnan(m['decay_hwhm_ms']) else
           f'FWHM={m["fwhm_ms"]:.1f} ms' if m else 'no response')
    print(f'  [{i_a+1}/{len(amp_levels)}] {amp:.2f} mW → '
          f'{amp_results[-1]["peak_mean_hz"]:.1f} ± {amp_results[-1]["peak_std_hz"]:.1f} Hz  {tag}')

print('Sweep A done.')

In [ ]:
cmap_a = plt.cm.plasma
a_norm = plt.Normalize(amp_levels.min(), amp_levels.max())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── PSTH traces ──────────────────────────────────────────────────────────
ax = axes[0]
for res in amp_results:
    col = cmap_a(a_norm(res['amp_mw']))
    ax.fill_between(t_fine, res['mean_hz'] - res['sem_hz'],
                    res['mean_hz'] + res['sem_hz'], color=col, alpha=0.15)
    ax.plot(t_fine, res['mean_hz'], color=col, lw=1.5)
ax.axvline(PULSE_ONSET_MS,              color='k', lw=0.8, ls='--', label='onset')
ax.axvline(PULSE_ONSET_MS + PULSE_DUR_MS, color='k', lw=0.8, ls=':', label='offset')
ax.set_xlabel('Time in stim window (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'Amplitude sweep  ({PULSE_DUR_MS:.0f} ms pulse, n={N_RUNS})')
ax.legend(frameon=False, fontsize=8)
sm_a = plt.cm.ScalarMappable(cmap=cmap_a, norm=a_norm)
sm_a.set_array([])
fig.colorbar(sm_a, ax=ax, label='Amplitude (mW)', shrink=0.85)

# ── Temporal precision vs peak rate ──────────────────────────────────────
ax = axes[1]
valid_a = [r for r in amp_results if r['metrics'] is not None]
pr      = [r['peak_mean_hz']           for r in valid_a]
fwhms   = [r['metrics']['fwhm_ms']        for r in valid_a]
rise_h  = [r['metrics']['rise_hwhm_ms']   for r in valid_a]
fall_h  = [r['metrics']['fall_hwhm_ms']   for r in valid_a]
dec_h   = [r['metrics']['decay_hwhm_ms']  for r in valid_a]

ax.plot(pr, fwhms,  'o-',  color='k',         lw=1.5, label='FWHM')
ax.plot(pr, rise_h, 's--', color='steelblue', lw=1.2, label='Rise HWHM')
ax.plot(pr, fall_h, '^--', color='tomato',    lw=1.2, label='Fall HWHM')
_vd = [(r, d) for r, d in zip(pr, dec_h) if not np.isnan(d)]
if _vd:
    _pr_d, _dec_d = zip(*_vd)
    ax.plot(_pr_d, _dec_d, 'D:', color='seagreen', lw=1.2, label='Decay HWHM (post-offset)')
ax.set_xlabel('Mean peak firing rate (Hz)')
ax.set_ylabel('Duration (ms)')
ax.set_title('Temporal precision vs evoked rate')
ax.legend(frameon=False, fontsize=8)

# ── Rate variability vs peak rate ─────────────────────────────────────────
ax = axes[2]
p_std = [r['peak_std_hz'] for r in valid_a]
ax.plot(pr, p_std, 'o-', color='purple', lw=1.5)
ax.set_xlabel('Mean peak firing rate (Hz)')
ax.set_ylabel('Std of peak rate across runs (Hz)')
ax.set_title('Rate variability vs evoked rate')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'precision_amp_sweep.png', dpi=150)
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
print(f'{"Amp (mW)":>9}  {"Peak (Hz)":>10}  {"±Std":>7}  '
      f'{"FWHM":>7}  {"Rise":>6}  {"Fall":>6}  {"Decay":>7}')
for res in amp_results:
    m = res['metrics']
    if m is None:
        print(f'{res["amp_mw"]:>9.2f}  — no response')
    else:
        print(f'{res["amp_mw"]:>9.2f}  {res["peak_mean_hz"]:>10.1f}  '
              f'{res["peak_std_hz"]:>7.1f}  '
              f'{m["fwhm_ms"]:>7.1f}  {m["rise_hwhm_ms"]:>6.1f}  '
              f'{m["fall_hwhm_ms"]:>6.1f}  '
              + (f'{m["decay_hwhm_ms"]:>7.1f}' if not np.isnan(m['decay_hwhm_ms']) else f'{"—":>7}'))

## Experiment B — Background current sweep

Varies `I_bg_exc_pA` to set different pre-existing baseline firing rates.
The probe pulse amplitude is fixed at `BG_PULSE_AMP_MW`.
Reveals whether the network's temporal and rate precision depend on how active it already is.

In [ ]:
print(f'Sweep B: {len(bg_levels)} background levels × {N_RUNS} runs  '
      f'(probe {BG_PULSE_AMP_MW:.1f} mW, {PULSE_DUR_MS:.0f} ms at t={PULSE_ONSET_MS:.0f} ms)')

_cfg_orig_bg = float(cfg.I_bg_exc_pA)
bg_results   = []
t_fine_bg    = None

for i_b, bg_pa in enumerate(bg_levels):
    cfg.I_bg_exc_pA = float(bg_pa)
    # Rebuild model with this background level and fine bins
    m_bg = RandomEINetwork(cfg, optics=_optics, power_curve=_curve,
                           normalization='max_expression')

    # Measure baseline rate (zero-amplitude pulse → pure spontaneous activity)
    _zero_wf = RectangularPulseWaveform(onset_ms=PULSE_ONSET_MS, duration_ms=PULSE_DUR_MS,
                                         amplitude=0.0)
    _r_base  = m_bg.run(_zero_wf)
    if t_fine_bg is None:
        t_fine_bg = _r_base['t_psth_ms']
    _base_psth = _r_base['psth_exc'] / (FINE_BIN_MS / 1000.0)
    pre_mask   = t_fine_bg < (PULSE_ONSET_MS - 15)
    baseline_hz = float(_base_psth[pre_mask].mean()) if pre_mask.any() else 0.0

    # Run pulse sweep
    wf = RectangularPulseWaveform(onset_ms=PULSE_ONSET_MS, duration_ms=PULSE_DUR_MS,
                                   amplitude=BG_PULSE_AMP_MW)
    _runs = []
    for _i in range(N_RUNS):
        _r = m_bg.run(wf, seed=SEED_BASE + _i,
                      vary_init_v=VARY_INIT_V,
                      vary_connectivity=VARY_CONNECTIVITY,
                      vary_weights=VARY_WEIGHTS)
        _runs.append(_r['psth_exc'] / (FINE_BIN_MS / 1000.0))

    _arr    = np.stack(_runs)
    mean_hz = _arr.mean(0)
    sem_hz  = _arr.std(0) / np.sqrt(N_RUNS)

    post_mask     = t_fine_bg >= PULSE_ONSET_MS
    per_run_peaks = _arr[:, post_mask].max(1) if post_mask.any() else np.zeros(N_RUNS)

    m = measure_pulse_response(t_fine_bg, mean_hz, PULSE_ONSET_MS, PULSE_DUR_MS)

    bg_results.append({
        'bg_pa':         bg_pa,
        'baseline_hz':   baseline_hz,
        'mean_hz':       mean_hz,
        'sem_hz':        sem_hz,
        'per_run_peaks': per_run_peaks,
        'peak_mean_hz':  float(per_run_peaks.mean()),
        'peak_std_hz':   float(per_run_peaks.std()),
        'metrics':       m,
    })
    tag = (f'FWHM={m["fwhm_ms"]:.1f} ms  decay={m["decay_hwhm_ms"]:.1f} ms'
           if m and not np.isnan(m['decay_hwhm_ms']) else
           f'FWHM={m["fwhm_ms"]:.1f} ms' if m else 'no response')
    print(f'  [{i_b+1}/{len(bg_levels)}] {bg_pa:.0f} pA  '
          f'baseline={baseline_hz:.1f} Hz → '
          f'{bg_results[-1]["peak_mean_hz"]:.1f} ± {bg_results[-1]["peak_std_hz"]:.1f} Hz  {tag}')

cfg.I_bg_exc_pA = _cfg_orig_bg  # restore
print('Sweep B done.')

In [ ]:
cmap_b = plt.cm.viridis
bl_rates = [r['baseline_hz'] for r in bg_results]
b_norm   = plt.Normalize(min(bl_rates) if bl_rates else 0, max(bl_rates) if bl_rates else 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── PSTH traces ──────────────────────────────────────────────────────────
ax = axes[0]
for res in bg_results:
    col = cmap_b(b_norm(res['baseline_hz']))
    ax.fill_between(t_fine_bg, res['mean_hz'] - res['sem_hz'],
                    res['mean_hz'] + res['sem_hz'], color=col, alpha=0.15)
    ax.plot(t_fine_bg, res['mean_hz'], color=col, lw=1.5)
ax.axvline(PULSE_ONSET_MS,                color='k', lw=0.8, ls='--', label='onset')
ax.axvline(PULSE_ONSET_MS + PULSE_DUR_MS, color='k', lw=0.8, ls=':',  label='offset')
ax.set_xlabel('Time in stim window (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'Background sweep  ({BG_PULSE_AMP_MW:.1f} mW probe, n={N_RUNS})')
ax.legend(frameon=False, fontsize=8)
sm_b = plt.cm.ScalarMappable(cmap=cmap_b, norm=b_norm)
sm_b.set_array([])
fig.colorbar(sm_b, ax=ax, label='Baseline firing rate (Hz)', shrink=0.85)

# ── Temporal precision vs baseline rate ──────────────────────────────────
ax = axes[1]
valid_b = [r for r in bg_results if r['metrics'] is not None]
bl      = [r['baseline_hz']              for r in valid_b]
fwhms_b = [r['metrics']['fwhm_ms']          for r in valid_b]
rise_b  = [r['metrics']['rise_hwhm_ms']     for r in valid_b]
fall_b  = [r['metrics']['fall_hwhm_ms']     for r in valid_b]
dec_b   = [r['metrics']['decay_hwhm_ms']    for r in valid_b]

ax.plot(bl, fwhms_b, 'o-',  color='k',         lw=1.5, label='FWHM')
ax.plot(bl, rise_b,  's--', color='steelblue', lw=1.2, label='Rise HWHM')
ax.plot(bl, fall_b,  '^--', color='tomato',    lw=1.2, label='Fall HWHM')
_vd_b = [(b, d) for b, d in zip(bl, dec_b) if not np.isnan(d)]
if _vd_b:
    _bl_d, _dec_d = zip(*_vd_b)
    ax.plot(_bl_d, _dec_d, 'D:', color='seagreen', lw=1.2, label='Decay HWHM (post-offset)')
ax.set_xlabel('Baseline firing rate (Hz)')
ax.set_ylabel('Duration (ms)')
ax.set_title('Temporal precision vs baseline rate')
ax.legend(frameon=False, fontsize=8)

# ── Rate variability vs baseline rate ────────────────────────────────────
ax = axes[2]
p_std_b = [r['peak_std_hz'] for r in valid_b]
ax.plot(bl, p_std_b, 'o-', color='purple', lw=1.5)
ax.set_xlabel('Baseline firing rate (Hz)')
ax.set_ylabel('Std of peak rate across runs (Hz)')
ax.set_title('Rate variability vs baseline rate')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'precision_bg_sweep.png', dpi=150)
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
print(f'{"Bg (pA)":>8}  {"Baseline":>9}  {"Peak (Hz)":>10}  {"±Std":>7}  '
      f'{"FWHM":>7}  {"Rise":>6}  {"Fall":>6}  {"Decay":>7}')
for res in bg_results:
    m = res['metrics']
    if m is None:
        print(f'{res["bg_pa"]:>8.0f}  {res["baseline_hz"]:>9.1f}  — no response')
    else:
        print(f'{res["bg_pa"]:>8.0f}  {res["baseline_hz"]:>9.1f}  '
              f'{res["peak_mean_hz"]:>10.1f}  {res["peak_std_hz"]:>7.1f}  '
              f'{m["fwhm_ms"]:>7.1f}  {m["rise_hwhm_ms"]:>6.1f}  '
              f'{m["fall_hwhm_ms"]:>6.1f}  '
              + (f'{m["decay_hwhm_ms"]:>7.1f}' if not np.isnan(m['decay_hwhm_ms']) else f'{"—":>7}'))

In [ ]:
import pickle
_save = {
    'pulse_onset_ms':   PULSE_ONSET_MS,
    'pulse_dur_ms':     PULSE_DUR_MS,
    'fine_bin_ms':      FINE_BIN_MS,
    'stim_dur_ms':      STIM_DUR_MS,
    'n_runs':           N_RUNS,
    'seed_base':        SEED_BASE,
    'n_exc':            N_EXC,
    'n_inh':            N_INH,
    'max_power_mw':     MAX_POWER_MW,
    # Sweep A
    'amp_levels_mw':    amp_levels,
    't_fine_ms':        t_fine,
    'amp_results':      amp_results,
    # Sweep B
    'bg_levels_pa':     bg_levels,
    'bg_pulse_amp_mw':  BG_PULSE_AMP_MW,
    't_fine_bg_ms':     t_fine_bg,
    'bg_results':       bg_results,
}
_save_path = OUTPUT_DIR / 'precision_results.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump(_save, f)
print(f'Saved → {_save_path}')